# ray.put() 是 Ray 框架中用于显式地将数据存入分布式对象存储（Object Store）的核心函数。

- <font color='green'>虽然我们在调用远程函数 task.remote(data) 时，Ray 会自动在后台处理数据的传输，</font>
- <font color='red'>但 ray.put() 赋予了开发者手动控制数据生命周期和存储位置的能力。它是实现高效分布式计算、特别是处理大数据集的关键工具。</font>

## 🎯 核心作用：为什么要用 ray.put()？
简单来说，<font color='red'>ray.put() 的作用是：“先把数据存到共享内存里，拿到一个‘取货凭证’，然后再把这个凭证分发给各个任务。”</font>

###  它主要有以下三大核心优势：

####    避免大数据的重复传输（一次写入，多次读取）
    
###### 如果你有一个非常大的数据集（例如 1GB 的 Pandas DataFrame 或 NumPy 数组），并且有多个任务需要处理它：

- 不使用 ray.put()：如果你直接在循环中 task.remote(big_data)，Ray 可能会尝试为每个任务序列化并传输一次数据，造成巨大的网络开销和内存浪费。
- <font color='red'>使用 ray.put()</font>：你先调用 data_ref = ray.put(big_data)，数据只会被传输并存储一次。随后，你只需将轻量级的 data_ref（对象引用）传递给任务，所有任务都能通过引用快速访问同一份数据。

####    实现“零拷贝”共享（Zero-Copy）
- Ray 的对象存储基于共享内存（Plasma）。<font color='red'>当你使用 ray.put() 存储数据后，位于同一节点上的不同 Worker 进程可以通过内存映射直接访问这块数据，而无需进行数据的复制。这对于内存敏感型应用至关重要。</font>

####    解耦数据生产与消费
- 你可以提前将数据准备好并存入存储，稍后再由不同的任务去读取。这使得数据加载和计算逻辑可以异步进行。

## ⚙️ 返回值：ObjectRef

ray.put(data) 不会返回数据本身，而是返回一个 ObjectRef（对象引用）。

- 你可以把它想象成“取货单”或“文件句柄”。
- 它是一个轻量级的对象，包含指向实际数据的唯一 ID。
- 你需要通过 ray.get(object_ref) 才能从这个引用中取回实际的数据。

## 🚀 典型应用场景与代码示例

### 场景一：多任务共享同一个大模型/大数据集
- 这是 ray.put() 最经典的用法。假设我们要用同一个大模型进行并行推理，或者用同一份训练数据进行多副本训练。

In [ ]:
import ray
import numpy as np

ray.init()

# 1. 模拟一个巨大的数据集 (例如 100MB)
big_data = np.random.rand(10000, 2500) 

# 2. 【关键步骤】显式将数据放入对象存储
# 数据只会被传输一次，存入共享内存
data_ref = ray.put(big_data)

@ray.remote
def compute_mean(data, factor):
    # 这里的 data 是通过引用获取的，效率极高
    return np.mean(data) * factor

# 3. 并行启动多个任务，传递的是“引用”而不是“数据本身”
# 就像给每个工人发了一张“仓库提货单”，而不是把货物搬来搬去
results = ray.get([
    compute_mean.remote(data_ref, i) 
    for i in range(10)
])

print(results)

### 场景二：提前加载大语言模型 (LLM)
- 在之前的对话中提到的 LLM 推理场景，<font color='red'>ray.put() 用于将加载好的模型权重存入存储，避免每个 Worker 重复加载模型。</font>


In [ ]:
# 假设 model 是加载好的庞大的 PyTorch 模型对象
# model_ref = ray.put(model) 

# 随后在推理任务中：
# @ray.remote
# def infer(model_ref, text):
#     model = ray.get(model_ref) # 获取模型引用
#     ...

## ⚠️ 关键特性与注意事项

1. 数据的不可变性 (Immutability)
- <font color='red'>Ray 对象存储中的数据是只读的。</font>
- 一旦你执行 obj_ref = ray.put([1, 2, 3])，存储在 Ray 中的这个列表就不能被修改。
- <font color='red'>如果你在任务中执行 ray.get(obj_ref) 并对其进行了修改（例如 .append(4)），你修改的只是当前进程内存中的副本，不会影响 Ray 对象存储中原始的数据。</font>

2. 内存管理与溢出 (Spilling)
- <font color='red'>内存压力：对象存储的大小是有限的（通常默认为系统内存的 30% 或指定大小）。</font>
- <font color='red'>溢出到磁盘：如果 ray.put() 存入的数据太多，填满了对象存储，Ray 会自动将旧的数据“溢出”（Spill）到磁盘上，腾出内存空间。这虽然能保证程序不崩溃，但会显著降低性能。</font>

3. 序列化开销

<b>虽然 ray.put() 避免了重复传输，但第一次存入时仍然需要序列化。对于某些极其复杂的对象，序列化本身可能很耗时。Ray 会自动选择最优的序列化器（如 Arrow 用于表格数据，Cloudpickle 用于通用对象）。</b>